# Data Quality Validation — a real, live generate → validate → repair run

This notebook re-runs the actual pipeline live against a local MySQL `hospital_db` instance and captures the real, unedited output — it is not a transcription of results from elsewhere.

**Requirements:** a local MySQL server (e.g. via XAMPP) with the `hospital_db` schema already loaded (`data/hospital_db.sql`), and `sqlalchemy`/`pymysql` installed.

**⚠️ This notebook truncates and regenerates `hospital_db`.** It uses `small` scale (800 admissions, ~30 seconds) so it runs quickly — the numbers here are therefore *not* the same run documented in [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md)'s results table (which used fresh small/medium/large runs each). They'll be in the same ballpark, since both use the same generator and the same fixed random seed, but small-scale runs have more sampling noise (see the seed-independent violation categories below).

If you have your own data in `hospital_db` you want to keep, **do not run this notebook** — or re-run `src/export_to_csv.py` against `large` scale afterward to restore the shipped dashboard data.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../data").resolve()))
sys.path.insert(0, str(Path("../src").resolve()))

# Both files are written for exec()-style Jupyter usage (see their own
# docstrings) rather than import, so we load them the same way here.
exec(open("../data/hospital_generator.py").read())
exec(open("../src/validate_repair.py").read())


## Generate a fresh `small`-scale database

In [2]:
main("small")



  Hospital Database Generator (constraint-aware)
  Config: SMALL â€” Small community hospital (~50 beds)



  All tables truncated.

[ Domain 1 ] Reference tables
  Inserting departments...
    20 departments inserted
  Inserting payers...
    10 payers inserted
  Inserting procedures...
    30 procedures inserted

[ Domain 2 ] Staff & Facilities
  Generating 500 patients...


    500 patients inserted
  Generating 25 doctors...
    Guaranteed doctor added for DEP-006
    Guaranteed doctor added for DEP-007
    Guaranteed doctor added for DEP-008
    Guaranteed doctor added for DEP-011
    Guaranteed doctor added for DEP-013
    Guaranteed doctor added for DEP-014
    Guaranteed doctor added for DEP-016
    Guaranteed doctor added for DEP-017
    Guaranteed doctor added for DEP-018
    Guaranteed doctor added for DEP-019
    35 doctors inserted
  Generating 60 nurses...
    60 nurses inserted
  Generating 40 employees...
    40 employees inserted
  Generating 55 rooms...
    55 rooms inserted

[ Domain 3 ] Clinical
  Generating 800 admissions...


    800 admissions inserted
  Generating triage for 663 admissions...
    663 triage records inserted
  Generating diagnoses...


    1341 diagnoses inserted
  Generating patient procedures...
    551 patient procedures inserted
  Generating medications...


    965 medications inserted
  Generating medical tests...
    771 medical tests inserted
  Generating staff shifts...
    1080 staff shifts inserted



[ Domain 4 ] Financial
  Generating billing records...
    646 billing records inserted
  Generating billing line items...


    2671 billing line items inserted

  Generation complete. Row counts:
  departments                          20 rows
  payers                               10 rows
  procedures                           30 rows
  patients                            500 rows
  doctors                              35 rows
  nurses                               60 rows
  employees                            40 rows
  rooms                                55 rows
  admissions                          800 rows
  triage                              663 rows
  diagnoses                         1,341 rows
  patient_procedures                  551 rows
  medications                         965 rows
  medical_tests                       771 rows
  staff_shifts                      1,080 rows
  billing                             646 rows
  billing_line_items                2,671 rows

  hospital_db is ready for analysis.



## Confirm the doctor-per-department guarantee

The generator originally had a real bug here (see [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md#a-fifth-bug-found-by-testing-at-small-scale)): at small scale, random specialty-based assignment left several departments with zero doctors, which made `doctor_dept_mismatch` violations permanently unrepairable. Confirming that never happens now:

In [3]:
engine = create_engine(DB_URL, echo=False)
with engine.connect() as conn:
    result = conn.execute(text('''
        SELECT d.dep_name, COUNT(doc.doctor_id) AS n_doctors
        FROM departments d
        LEFT JOIN doctors doc ON doc.department_id = d.department_id
        GROUP BY d.dep_name
        ORDER BY n_doctors ASC
    '''))
    rows = result.fetchall()

zero_doctor_depts = [r[0] for r in rows if r[1] == 0]
print(f"Departments with zero doctors: {len(zero_doctor_depts)}")
print(f"Minimum doctors in any department: {min(r[1] for r in rows)}")
for name, n in rows[:5]:
    print(f"  {name:<25} {n} doctors")


Departments with zero doctors: 0
Minimum doctors in any department: 1
  Orthopedic                1 doctors
  Cardiology                1 doctors
  Hematology                1 doctors
  Anesthesiology            1 doctors
  Urology                   1 doctors


## Pre-repair validation

In [4]:
with engine.connect() as conn:
    before = validate_all(conn)
print_report(before)



  Validation Report

  Group 1 â€” Temporal â€” âœ“ clean
        âœ“  discharge_before_admission                         0
        âœ“  los_inconsistent                                   0
        âœ“  triage_out_of_window                               0
        âœ“  diagnosis_out_of_window                            0
        âœ“  procedure_out_of_window                            0
        âœ“  medication_date_issues                             0
        âœ“  test_out_of_window                                 0
        âœ“  result_before_test                                 0
        âœ“  bill_before_discharge                              0
        âœ“  payment_before_bill                                0

  Group 2 â€” Age-based â€” âœ“ clean
        âœ“  newborn_wrong_age                                  0
        âœ“  newborn_wrong_dept                                 0
        âœ“  newborn_wrong_registered_date                      0
        âœ“  pediatric_in_geriatric_dept    

## Run repairs

In [5]:
with engine.connect() as conn:
    repair_all(conn)



  Running repairs...

  [Group 1] Repairing temporal violations...


    Registered dates adjusted for 500 patients
    Group 1 complete.
  [Group 2] Repairing age-based violations...
    Fixed birth/age/registered for 0 newborn patients
    Resampled 0 impossible age-diagnosis combinations
    Group 2 complete.
  [Group 3] Repairing admission type violations...
    Group 3 complete.
  [Group 4] Repairing department consistency violations...
    Reassigned doctors for 0 admissions
    Reassigned rooms for 0 admissions


    Fixed condition/room for 148 admissions (0 room reassignments, 148 condition adjustments)
    Group 4 complete.
  [Group 5] Repairing diagnosis alignment violations...
    Resampled 0 male-impossible, 0 female-impossible, 0 newborn-on-adult diagnoses
    Group 5 complete.
  [Group 6] Repairing financial violations...
    Group 6 complete.

  All repairs complete.



## Post-repair validation

In [6]:
with engine.connect() as conn:
    after = validate_all(conn)
print_report(after)



  Validation Report

  Group 1 â€” Temporal â€” âœ“ clean
        âœ“  discharge_before_admission                         0
        âœ“  los_inconsistent                                   0
        âœ“  triage_out_of_window                               0
        âœ“  diagnosis_out_of_window                            0
        âœ“  procedure_out_of_window                            0
        âœ“  medication_date_issues                             0
        âœ“  test_out_of_window                                 0
        âœ“  result_before_test                                 0
        âœ“  bill_before_discharge                              0
        âœ“  payment_before_bill                                0

  Group 2 â€” Age-based â€” âœ“ clean
        âœ“  newborn_wrong_age                                  0
        âœ“  newborn_wrong_dept                                 0
        âœ“  newborn_wrong_registered_date                      0
        âœ“  pediatric_in_geriatric_dept    

In [7]:
before_total = sum(sum(g.values()) for g in before.values())
after_total = sum(sum(g.values()) for g in after.values())
pct_eliminated = (1 - after_total / before_total) * 100 if before_total else 0

print(f"Before repair: {before_total:,} violations")
print(f"After repair:  {after_total:,} violations")
print(f"Eliminated:    {pct_eliminated:.1f}% in a single repair pass, no regenerate loop")


Before repair: 1,504 violations
After repair:  54 violations
Eliminated:    96.4% in a single repair pass, no regenerate loop


## Takeaways

- Zero departments end up with no doctors — the guarantee added to `gen_doctors()` holds, and `doctor_dept_mismatch` is fully repairable as a result.
- The vast majority of violations are eliminated in a single repair pass, matching the pattern documented at all three scales in `docs/METHODOLOGY.md`.
- Any violations remaining after repair are the two disclosed structural edge cases (room-type scarcity in thin departments, and the rare double-`Newborn`-admission case) — not bugs, and not something a repair loop would ever resolve by re-running.

## Restoring the shipped large-scale dataset

This notebook leaves `hospital_db` at `small` scale. To restore it to the `large`, repaired state that `app/data/` was exported from:

```python
main("large")
run_repair()
```
then re-run `python src/export_to_csv.py` from the repo root.